In [ ]:
# %pip install langchain-text-splitters nltk sentence-transformers tiktoken


In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

In [ ]:
# %pip install langchain

In [ ]:
# Install required packages
# pip install langchain-text-splitters nltk sentence-transformers tiktoken

import re
from typing import List, Dict
import nltk
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK data for sentence splitting
nltk.download('punkt')

# Sample text for demonstration
sample_text = """
Artificial Intelligence (AI) is transforming how we interact with technology. 
Machine learning, a subset of AI, enables computers to learn from data without explicit programming.

Deep learning uses neural networks with multiple layers. These networks can learn hierarchical representations of data. Natural Language Processing (NLP) allows computers to understand human language.

There are several types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. Each has different applications and use cases.

Supervised learning uses labeled data to train models. The algorithm learns to map inputs to outputs based on example input-output pairs.
Unsupervised learning finds patterns in unlabeled data. Common techniques include clustering and dimensionality reduction.
Reinforcement learning trains agents through rewards and punishments in an environment.
"""

1. Naive/Fixed-Size Chunking

In [ ]:
def naive_chunking(text: str, chunk_size: int = 100, chunk_overlap: int = 20) -> List[str]:
    """
    Simple fixed-size chunking with overlap
    """
    chunks = []
    start = 0
    text_length = len(text)
    
    while start < text_length:
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        
        # Move forward by chunk_size minus overlap
        start += chunk_size - chunk_overlap
        
        # Stop if we're not making progress
        if start >= text_length:
            break
            
    return chunks

# Test naive chunking
naive_chunks = naive_chunking(sample_text, chunk_size=150, chunk_overlap=30)
print("=== Naive Chunking ===")
for i, chunk in enumerate(naive_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}...")

2. Sentence-Based Chunking

In [ ]:
def sentence_chunking(text: str, sentences_per_chunk: int = 3) -> List[str]:
    """
    Chunk by grouping fixed number of sentences
    """
    # Split into sentences using NLTK
    sentences = nltk.sent_tokenize(text)
    
    chunks = []
    for i in range(0, len(sentences), sentences_per_chunk):
        chunk = ' '.join(sentences[i:i + sentences_per_chunk])
        chunks.append(chunk)
    
    return chunks

# Test sentence chunking
sentence_chunks = sentence_chunking(sample_text, sentences_per_chunk=2)
print("\n=== Sentence Chunking ===")
for i, chunk in enumerate(sentence_chunks):
    print(f"Chunk {i+1}: {chunk}")

3. Recursive Chunking

In [ ]:
def recursive_chunking(text: str, chunk_size: int = 150, separators: List[str] = None) -> List[str]:
    """
    Hierarchical chunking using multiple separators
    """
    if separators is None:
        separators = ["\n\n", "\n", ". ", " ", ""]
    
    def split_text(current_text, separator_index=0):
        if separator_index >= len(separators):
            return [current_text]
        
        separator = separators[separator_index]
        if separator == "":
            # Final fallback: split by character length
            return [current_text[i:i+chunk_size] for i in range(0, len(current_text), chunk_size)]
        
        if separator in current_text:
            splits = current_text.split(separator)
            chunks = []
            current_chunk = ""
            
            for split in splits:
                # If adding this split would make the chunk too large, save current chunk and start new one
                if len(current_chunk) + len(split) > chunk_size and current_chunk:
                    chunks.append(current_chunk.strip())
                    current_chunk = split
                else:
                    if current_chunk:
                        current_chunk += separator + split
                    else:
                        current_chunk = split
            
            if current_chunk:
                chunks.append(current_chunk.strip())
            
            # Further split chunks that are still too large
            final_chunks = []
            for chunk in chunks:
                if len(chunk) > chunk_size:
                    final_chunks.extend(split_text(chunk, separator_index + 1))
                else:
                    final_chunks.append(chunk)
            
            return final_chunks
        else:
            return split_text(current_text, separator_index + 1)
    
    return split_text(text)

# Test recursive chunking
recursive_chunks = recursive_chunking(sample_text, chunk_size=120)
print("\n=== Recursive Chunking ===")
for i, chunk in enumerate(recursive_chunks):
    print(f"Chunk {i+1} ({len(chunk)} chars): {chunk}")

4. Semantic Chunking

In [ ]:
class SemanticSplitter:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
    
    def semantic_chunking(self, text: str, similarity_threshold: float = 0.5) -> List[str]:
        """
        Split text based on semantic similarity between sentences
        """
        # Split into sentences
        sentences = nltk.sent_tokenize(text)
        
        if len(sentences) <= 1:
            return [text]
        
        # Get sentence embeddings
        embeddings = self.model.encode(sentences)
        
        # Calculate similarities between consecutive sentences
        chunks = []
        current_chunk = [sentences[0]]
        
        for i in range(1, len(sentences)):
            # Calculate cosine similarity between current and previous sentence
            similarity = cosine_similarity(
                [embeddings[i-1]], 
                [embeddings[i]]
            )[0][0]
            
            if similarity >= similarity_threshold:
                # Similar enough, add to current chunk
                current_chunk.append(sentences[i])
            else:
                # Significant topic shift, start new chunk
                chunks.append(' '.join(current_chunk))
                current_chunk = [sentences[i]]
        
        # Add the last chunk
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks

# Test semantic chunking
splitter = SemanticSplitter()
semantic_chunks = splitter.semantic_chunking(sample_text, similarity_threshold=0.3)
print("\n=== Semantic Chunking ===")
for i, chunk in enumerate(semantic_chunks):
    print(f"Chunk {i+1}: {chunk}")

5. Markdown-Aware Chunking

In [ ]:
def markdown_chunking(markdown_text: str) -> List[Dict]:
    """
    Chunk Markdown content by headings
    """
    # Split by markdown headings (##, ###, etc.)
    heading_pattern = r'(#+\s+.+)'
    sections = re.split(heading_pattern, markdown_text)
    
    chunks = []
    current_heading = "Root"
    
    for i, section in enumerate(sections):
        if re.match(heading_pattern, section):
            current_heading = section.strip()
        else:
            if section.strip():  # Skip empty sections
                chunks.append({
                    'heading': current_heading,
                    'content': section.strip(),
                    'full_text': f"{current_heading}\n{section.strip()}"
                })
    
    return chunks

# Test with markdown-like content
markdown_sample = """
# Machine Learning
Machine learning is a subset of artificial intelligence.

## Supervised Learning
Supervised learning uses labeled data to train models.

## Unsupervised Learning 
Unsupervised learning finds patterns in unlabeled data.
"""

markdown_chunks = markdown_chunking(markdown_sample)
print("\n=== Markdown Chunking ===")
for i, chunk in enumerate(markdown_chunks):
    print(f"Chunk {i+1} - {chunk['heading']}: {chunk['content']}")

6. Using LangChain Chunkers

In [ ]:
# from langchain.text_splitter import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

def langchain_chunking_example(text: str):
    """
    Demonstrate using LangChain's built-in chunkers
    """
    
    # Recursive character text splitter (recommended)
    recursive_splitter = RecursiveCharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    recursive_chunks = recursive_splitter.split_text(text)
    
    # Character text splitter (naive approach)
    char_splitter = CharacterTextSplitter(
        chunk_size=100,
        chunk_overlap=20,
        length_function=len
    )
    
    char_chunks = char_splitter.split_text(text)
    
    return recursive_chunks, char_chunks

# Test LangChain chunkers
recursive_lc, char_lc = langchain_chunking_example(sample_text)
print("\n=== LangChain Recursive Chunking ===")
for i, chunk in enumerate(recursive_lc):
    print(f"Chunk {i+1}: {chunk[:60]}...")

Comprehensive Comparison Function

In [ ]:
def compare_chunking_strategies(text: str):
    """
    Compare all chunking strategies on the same text
    """
    print("COMPARING CHUNKING STRATEGIES")
    print("=" * 50)
    
    # Test all strategies
    strategies = {
        "Naive (100 chars)": naive_chunking(text, chunk_size=100, chunk_overlap=20),
        "Sentence (2 sent/chunk)": sentence_chunking(text, sentences_per_chunk=2),
        "Recursive (120 chars)": recursive_chunking(text, chunk_size=120),
        "Semantic (threshold=0.3)": splitter.semantic_chunking(text, similarity_threshold=0.3)
    }
    
    for name, chunks in strategies.items():
        print(f"\n{name}:")
        print(f"  Number of chunks: {len(chunks)}")
        print(f"  Avg chunk length: {np.mean([len(chunk) for chunk in chunks]):.1f} chars")
        print(f"  Sample chunks:")
        for i, chunk in enumerate(chunks[:2]):  # Show first 2 chunks
            print(f"    {i+1}. {chunk[:50]}...")

# Run comparison
compare_chunking_strategies(sample_text)

Key Takeaways from the Code:
1. Naive Chunking: Simple but can break sentences and meaning

2. Sentence Chunking: Preserves sentence integrity but fixed grouping

3. Recursive Chunking: Intelligent hierarchical approach (often the best default)

4. Semantic Chunking: Most sophisticated but requires ML model

5. Markdown Chunking: Structure-aware for formatted documents

Practical Recommendations:

In [ ]:
def get_recommended_chunker(doc_type: str):
    """
    Suggest chunking strategy based on document type
    """
    recommendations = {
        "general_text": "Use RecursiveCharacterTextSplitter from LangChain",
        "code": "Use language-specific splitters (e.g., by functions/classes)",
        "markdown": "Use heading-aware chunking",
        "conversations": "Use semantic chunking with low similarity threshold",
        "scientific_papers": "Use section-based chunking"
    }
    return recommendations.get(doc_type, "Use recursive chunking as default")

# Example usage
print(get_recommended_chunker("general_text"))